# ASR experiment for TrustworthyRAG (Colab, free T4 GPU)
Set **Runtime -> Change runtime type -> T4 GPU**, then run the cells in order.
Edit `REPO_URL` in cell 2 to your own copy of this repository. After a runtime reset `/content` is wiped: re-run cells 1-4 (results are safe only if you ran the backup cell 9).

In [ ]:
# 1. GPU check
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2. Clone this repo and the upstream TrustworthyRAG code, then install dependencies
import os
REPO_URL = "https://github.com/YOUR_USERNAME/rag-poisoning-asr"   # <- edit
!rm -rf /content/rag-poisoning-asr && git clone -q {REPO_URL} /content/rag-poisoning-asr
os.chdir("/content/rag-poisoning-asr")
!bash scripts/setup_upstream.sh
os.environ["TRUSTWORTHYRAG_PATH"] = "/content/rag-poisoning-asr/TrustworthyRAG"
!pip install -q -r requirements.txt
print("cwd:", os.getcwd())

In [ ]:
# 3. Ollama + model (needed again after every runtime reset; takes a few minutes)
!apt-get install -y -q zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull mistral:7b

In [ ]:
# 4. Warm the model onto the GPU (the first request is slow and can exceed a 120 s timeout), then get TruthfulQA
import requests
r = requests.post("http://localhost:11434/v1/completions",
                  json={"model": "mistral:7b", "prompt": "Hello", "max_tokens": 1}, timeout=900)
print("warm-up status:", r.status_code)
!ollama ps
!python scripts/download_truthfulqa.py

In [ ]:
# 5. SMOKE TEST (6 queries): check the printed labels look sane before the full run
import os; os.chdir("/content/rag-poisoning-asr")
os.environ["TRUSTWORTHYRAG_PATH"] = "/content/rag-poisoning-asr/TrustworthyRAG"
!rm -f data/asr/smoke.jsonl
!python asr_experiment.py --n 6 --k 3 --out data/asr/smoke.jsonl
!python analyze_asr.py --in data/asr/smoke.jsonl --outdir data/asr/smoke

In [ ]:
# 6. FULL RUN: conditions A, B, G, R and C. Resumable: if it stops, run this cell again.
!python asr_experiment.py --n 100 --k 3 --poison-ratio 0.5 --strategies replace inject contradict --ref --judge --out data/asr/asr_results.jsonl

In [ ]:
# 7. Post hoc NLI reference conditions: R2 (top-K) then R3 (whole corpus). Needs Ollama warm (cell 4).
!python rerun_ref_nli.py --in data/asr/asr_results.jsonl --out data/asr/asr_results_r2.jsonl
!python rerun_ref_nli.py --in data/asr/asr_results_r2.jsonl --out data/asr/asr_results_r3.jsonl --mode corpus --tag R3

In [ ]:
# 8. Tables, statistics and chart
!python analyze_asr.py --in data/asr/asr_results_r3.jsonl --outdir data/asr/summary
!python stats_asr.py --in data/asr/asr_results_r3.jsonl
from IPython.display import Image
Image("data/asr/summary/asr_chart.png")

In [ ]:
# 9. BACKUP: download all results to your computer (run after every big batch)
!zip -rq /content/asr_backup.zip data/asr
from google.colab import files
files.download("/content/asr_backup.zip")